## 1. Install packages
(torch, pandas and scikit-learn are already in Colab.)

In [1]:
!pip -q install imbalanced-learn xgboost statsmodels scipy

## 2. Upload the two data files
Select: `corrected_unitstay_level_features_eicu_v2.csv` and the eICU demo zip (`eicu-collaborative-research-database-demo-2_0_1_1.zip`).

In [2]:
import os, zipfile
from google.colab import files
os.makedirs('/content/data', exist_ok=True)
up = files.upload()
for n in up:
    if n.endswith('.csv'): os.replace(n, '/content/data/'+n)
    elif n.endswith('.zip'): zipfile.ZipFile(n).extractall('/content/data')
print(os.listdir('/content/data'))

Saving eicu-collaborative-research-database-demo-2.0.1 2.zip to eicu-collaborative-research-database-demo-2.0.1 2.zip
['eicu-collaborative-research-database-demo-2.0.1']


In [3]:
import os, zipfile
from google.colab import files
os.makedirs('/content/data', exist_ok=True)
up = files.upload()
for n in up:
    if n.endswith('.csv'): os.replace(n, '/content/data/'+n)
    elif n.endswith('.zip'): zipfile.ZipFile(n).extractall('/content/data')
print(os.listdir('/content/data'))

Saving corrected_unitstay_level_features_eicu_v2.csv to corrected_unitstay_level_features_eicu_v2.csv
['corrected_unitstay_level_features_eicu_v2.csv', 'eicu-collaborative-research-database-demo-2.0.1']


## 3. Settings (paths are found automatically)

In [4]:
import os, glob, zipfile, sys
from pathlib import Path
RAW = Path(os.path.dirname(glob.glob('/content/data/**/patient.csv.gz', recursive=True)[0]))
EICU_RAW_DIR = RAW
FEATURES_CSV = Path(glob.glob('/content/data/**/corrected_unitstay_level_features_eicu_v2.csv', recursive=True)[0])
OUT_DIR = Path('/content/outputs'); OUT_DIR.mkdir(exist_ok=True)
N_REPEATS = 3        # repeats of 5-fold CV (paper: 3). Use 1 for a quick test run.
USE_MLP = True       # neural network (PyTorch)
print('eICU raw folder :', EICU_RAW_DIR); print('features csv    :', FEATURES_CSV); print('N_REPEATS =', N_REPEATS)

eICU raw folder : /content/data/eicu-collaborative-research-database-demo-2.0.1
features csv    : /content/data/corrected_unitstay_level_features_eicu_v2.csv
N_REPEATS = 3


## 4. Shared code: data, feature sets, models (including the PyTorch MLP), cross-validation helpers

In [5]:
import pandas as pd, numpy as np, warnings
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
warnings.filterwarnings('ignore')
try:
    import torch, torch.nn as nn
    HAVE_TORCH = True
except Exception:
    HAVE_TORCH = False
MODELS = ['LR','RF','XGB'] + (['MLP'] if (USE_MLP and HAVE_TORCH) else [])
if USE_MLP and not HAVE_TORCH: print("WARNING: torch not installed -> MLP skipped (pip install torch)")

f = pd.read_csv(FEATURES_CSV)
f['age'] = pd.to_numeric(f['age'].replace('> 89','90'), errors='coerce')
y = f.Target.astype(int).values; g = f.uniquepid.values
demo = ['age','gender','admissionweight']
val = [c for c in f.columns if c.endswith('_mean')]
ind = [c for c in f.columns if c.endswith('_measured') or c.endswith('_given')]
SETS = {'demographics':demo,'values_only':demo+val,'indicators_only':demo+ind,'both':demo+val+ind,'ind_no_demo':ind}

class MLPNet(nn.Module if HAVE_TORCH else object):
    def __init__(s,d):
        super().__init__()
        s.net=nn.Sequential(nn.Linear(d,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(0.4),
                            nn.Linear(32,16),nn.BatchNorm1d(16),nn.ReLU(),nn.Dropout(0.4),nn.Linear(16,1))
    def forward(s,x): return s.net(x).squeeze(-1)

def fit_mlp(Xa,Xb,ya,ga,seed,pos_weight=True,protocol='es15',max_epochs=100,patience=10,fixed_epochs=30):
    """MLP 32-16, BatchNorm, dropout 0.4, AdamW wd=1e-4, lr=1e-3, batch 64.
    protocol 'es15'  : 15% inner patient-grouped validation subset, early stopping (primary protocol of the paper)
    protocol 'fixed' : no validation subset, fixed number of epochs (matched-protocol check)"""
    torch.manual_seed(seed); np.random.seed(seed)
    Xa_=torch.tensor(Xa,dtype=torch.float32); ya_=torch.tensor(ya,dtype=torch.float32); Xb_=torch.tensor(Xb,dtype=torch.float32)
    pw=torch.tensor([(ya==0).sum()/max((ya==1).sum(),1)],dtype=torch.float32) if pos_weight else None
    lossf=nn.BCEWithLogitsLoss(pos_weight=pw)
    m=MLPNet(Xa.shape[1]); opt=torch.optim.AdamW(m.parameters(),lr=1e-3,weight_decay=1e-4)
    def epoch(idx):
        m.train(); perm=np.random.permutation(idx)
        for i in range(0,len(perm),64):
            b=perm[i:i+64]
            if len(b)<2: continue
            opt.zero_grad(); l=lossf(m(Xa_[b]),ya_[b]); l.backward(); opt.step()
    if protocol=='fixed':
        for _ in range(fixed_epochs): epoch(np.arange(len(ya)))
    else:
        sp=StratifiedGroupKFold(7,shuffle=True,random_state=seed)   # first fold ~ 14-15% of patients
        tr,va=next(sp.split(Xa,ya,ga))
        best=1e9;bad=0;state=None
        for _ in range(max_epochs):
            epoch(tr); m.eval()
            with torch.no_grad(): vl=lossf(m(Xa_[va]),ya_[va]).item()
            if vl<best-1e-4: best=vl;bad=0;state={k:v.clone() for k,v in m.state_dict().items()}
            else:
                bad+=1
                if bad>=patience: break
        if state is not None: m.load_state_dict(state)
    m.eval()
    with torch.no_grad(): return torch.sigmoid(m(Xb_)).numpy()

def mk(name,seed=0):
    if name=='LR': return LogisticRegression(C=0.5,max_iter=2000,class_weight='balanced')
    if name=='RF': return RandomForestClassifier(200,max_depth=5,min_samples_leaf=15,class_weight='balanced',n_jobs=-1,random_state=seed)
    if name=='XGB': return XGBClassifier(n_estimators=150,max_depth=3,learning_rate=0.03,scale_pos_weight=(y==0).sum()/(y==1).sum(),n_jobs=-1,random_state=seed,verbosity=0)

def folds(R=None,K=5):
    R=R or N_REPEATS
    for r in range(R):
        for k,(a,b) in enumerate(StratifiedGroupKFold(K,shuffle=True,random_state=42+r).split(f,y,g)): yield r,k,a,b

def prep(cols,a,b,mode='train_median',rng=None):
    Xa=f.loc[f.index[a],cols].astype(float).copy(); Xb=f.loc[f.index[b],cols].astype(float).copy()
    if mode=='train_median': m=Xa.median(); Xa=Xa.fillna(m); Xb=Xb.fillna(m)
    elif mode=='test_median': Xa=Xa.fillna(Xa.median()); Xb=Xb.fillna(Xb.median())
    elif mode=='random_draw':
        for c in cols:
            obs=Xa[c].dropna().values
            if len(obs)==0: continue
            for X in (Xa,Xb):
                na=X[c].isna(); X.loc[na,c]=rng.choice(obs,na.sum())
    return Xa.values,Xb.values

def fit_pred(name,Xa,Xb,ya,res='weights',seed=0,ga=None,mlp_protocol='es15'):
    sc=StandardScaler().fit(Xa); Xa=sc.transform(Xa); Xb=sc.transform(Xb)
    if name=='MLP':
        if res=='smote': Xa,ya,ga=*SMOTE(random_state=seed).fit_resample(Xa,ya),None  # inner grouped split not possible after SMOTE
        elif res=='under':
            rus=RandomUnderSampler(random_state=seed); Xa,ya=rus.fit_resample(Xa,ya); ga=ga[rus.sample_indices_]
        if ga is None: ga=np.arange(len(ya))
        return fit_mlp(Xa,Xb,ya,ga,seed,pos_weight=(res=='weights'),protocol=mlp_protocol)
    m=mk(name,seed)
    if res=='none':
        if name in('LR','RF'): m.set_params(class_weight=None)
        if name=='XGB': m.set_params(scale_pos_weight=1)
    elif res=='smote': Xa,ya=SMOTE(random_state=seed).fit_resample(Xa,ya); m.set_params(**({'class_weight':None} if name!='XGB' else {'scale_pos_weight':1}))
    elif res=='under': Xa,ya=RandomUnderSampler(random_state=seed).fit_resample(Xa,ya); m.set_params(**({'class_weight':None} if name!='XGB' else {'scale_pos_weight':1}))
    m.fit(Xa,ya); return m.predict_proba(Xb)[:,1]

def run(tag,name,cols,res='weights',mode='train_median',R=None,mlp_protocol='es15'):
    rows=[]; rng=np.random.default_rng(1)
    for r,k,a,b in folds(R):
        Xa,Xb=prep(cols,a,b,mode,rng); pr=fit_pred(name,Xa,Xb,y[a],res,seed=r,ga=g[a],mlp_protocol=mlp_protocol)
        rows.append(dict(tag=tag,model=name,rep=r,fold=k,auroc=roc_auc_score(y[b],pr),auprc=average_precision_score(y[b],pr),
          bacc=balanced_accuracy_score(y[b],pr>0.5),auroc_hard=roc_auc_score(y[b],(pr>0.5).astype(int)),ntr=len(a),nte=len(b),pos_te=int(y[b].sum()),pos_tr=int(y[a].sum()),
          pts_te=len(set(g[b])),overlap=len(set(g[a])&set(g[b]))))
    return pd.DataFrame(rows)

print('Models used:', MODELS)
print('Features:', f.shape, ' positives:', y.sum())

Models used: ['LR', 'RF', 'XGB', 'MLP']
Features: (2155, 57)  positives: 220


## 5. Step 1 – label components and timing of the first diagnosis entry (seconds)

In [6]:
"""Rebuild label components + timing of first matching diagnosis (reviewer comments R2.1, R2.2)."""
p=pd.read_csv(EICU_RAW_DIR/'patient.csv.gz'); d=pd.read_csv(EICU_RAW_DIR/'diagnosis.csv.gz')
f1=pd.read_csv(FEATURES_CSV)
d['codes']=d.icd9code.fillna('').str.replace('.','',regex=False).str.upper().str.split(r'\s*,\s*')
def comp(r):
    cs=set(c.strip() for c in r.codes); s=str(r.diagnosisstring).lower(); out=[]
    if cs&{'2930','2931','F05','F050','F051'} or 'delirium' in s: out.append('delirium')
    if cs&{'34830','G9340','34831','G9341'}: out.append('encephalopathy')
    if cs&{'34982','G92'}: out.append('toxic_enceph')
    if cs&{'3481','G931'}: out.append('anoxic')
    if cs&{'78009','R4182'}: out.append('AMS')
    return out
d['comps']=d.apply(comp,axis=1); dd=d[d.comps.str.len()>0].copy()
pos=set(dd.patientunitstayid); fpos=set(f1.loc[f1.Target,'patientunitstayid'])
print('positives rebuilt:',len(pos),' in feature table:',len(fpos),' identical:',pos==fpos)
x=dd.explode('comps'); print(x.groupby('comps').patientunitstayid.nunique())
first=dd.groupby('patientunitstayid').diagnosisoffset.min()
los=p.set_index('patientunitstayid').unitdischargeoffset
print(first.describe()); print('<=0:',(first<=0).sum(),' 1-1440:',((first>0)&(first<=1440)).sum(),' >1440:',(first>1440).sum(),' LOS<24h among positives:',(los[first.index]<1440).sum())
first.rename('first_dx_offset').to_csv(OUT_DIR/'first_offsets.csv')
rows=[]
for c,gp in x.groupby('comps'):
    fo=gp.groupby('patientunitstayid').diagnosisoffset.min(); rows.append(dict(component=c,stays=len(fo),first_le_1440=int((fo<=1440).sum()),first_gt_1440=int((fo>1440).sum())))
pd.DataFrame(rows).to_csv(OUT_DIR/'label_components.csv',index=False)
print('stays',len(f1),'patients',f1.uniquepid.nunique(),'multi-stay patients',(f1.groupby('uniquepid').size()>1).sum(),'positive patients',f1[f1.Target].uniquepid.nunique())
print('hospital ids in patient table:',p.hospitalid.nunique())


positives rebuilt: 220  in feature table: 220  identical: True
comps
AMS               169
anoxic              6
delirium           20
encephalopathy     46
toxic_enceph        8
Name: patientunitstayid, dtype: int64
count     220.000000
mean      544.386364
std      1275.052944
min      -108.000000
25%        34.000000
50%        94.500000
75%       257.000000
max      8885.000000
Name: diagnosisoffset, dtype: float64
<=0: 7  1-1440: 191  >1440: 22  LOS<24h among positives: 61
stays 2155 patients 1709 multi-stay patients 285 positive patients 206
hospital ids in patient table: 186


## 6. Step 2 – ablation, SMOTE/undersampling, imputation, record-level split, MLP protocol check  (**longest step**)
One line is printed each time a model/setting finishes all folds.

In [7]:
"""Ablation (values/indicators/both), SMOTE/undersampling, imputation, record-level split, MLP protocol check."""
from sklearn.model_selection import StratifiedKFold
allc=SETS['both']; out=[]
for m in MODELS:
    for s,c in SETS.items():
        out.append(run('abl_'+s,m,c)); print('ablation',m,s,round(out[-1].auroc.mean(),3),flush=True)
pd.concat(out).to_csv(OUT_DIR/'abl.csv',index=False)
out=[]
for m in MODELS:
    for res in ['none','weights','smote','under']:
        out.append(run('res_'+res,m,allc,res=res)); print('resampling',m,res,round(out[-1].auroc.mean(),3),round(out[-1].auprc.mean(),3),flush=True)
for mode in ['train_median','test_median','random_draw']:
    for m in ['LR','XGB']:
        for s in ['values_only','both']:
            out.append(run(f'imp_{mode}_{s}',m,SETS[s],mode=mode)); print('imputation',mode,m,s,round(out[-1].auroc.mean(),3),flush=True)
pd.concat(out).to_csv(OUT_DIR/'res_imp.csv',index=False)
if 'MLP' in MODELS:   # reviewer R1.12: same protocol for all models
    r=[run('mlp_es15','MLP',allc,mlp_protocol='es15'),run('mlp_fixed','MLP',allc,mlp_protocol='fixed')]
    r=pd.concat(r); r.to_csv(OUT_DIR/'mlp_protocol_runs.csv',index=False); print(r.groupby('tag')[['auroc','auprc']].agg(['mean','std']))
rows=[]
for r in range(N_REPEATS):
    for k,(a,b) in enumerate(StratifiedKFold(5,shuffle=True,random_state=42+r).split(f,y)):
        Xa,Xb=prep(allc,a,b)
        for m in MODELS:
            pr=fit_pred(m,Xa,Xb,y[a],'weights',r,ga=g[a]); rows.append(dict(model=m,auroc=roc_auc_score(y[b],pr),overlap=len(set(g[a])&set(g[b])),design='record_level'))
rl=pd.DataFrame(rows); rl.to_csv(OUT_DIR/'recordlevel.csv',index=False); print(rl.groupby('model')[['auroc','overlap']].mean())


ablation LR demographics 0.511
ablation LR values_only 0.568
ablation LR indicators_only 0.608
ablation LR both 0.608
ablation LR ind_no_demo 0.616
ablation RF demographics 0.509
ablation RF values_only 0.586
ablation RF indicators_only 0.578
ablation RF both 0.594
ablation RF ind_no_demo 0.591
ablation XGB demographics 0.493
ablation XGB values_only 0.57
ablation XGB indicators_only 0.569
ablation XGB both 0.565
ablation XGB ind_no_demo 0.598
ablation MLP demographics 0.528
ablation MLP values_only 0.557
ablation MLP indicators_only 0.571
ablation MLP both 0.6
ablation MLP ind_no_demo 0.576
resampling LR none 0.609 0.18
resampling LR weights 0.608 0.181
resampling LR smote 0.599 0.173
resampling LR under 0.586 0.158
resampling RF none 0.611 0.186
resampling RF weights 0.594 0.181
resampling RF smote 0.58 0.172
resampling RF under 0.585 0.165
resampling XGB none 0.585 0.178
resampling XGB weights 0.565 0.163
resampling XGB smote 0.576 0.164
resampling XGB under 0.573 0.155
resampling M

## 7. Step 3 – paired Nadeau–Bengio tests, per-fold class counts, hard-label AUROC

In [8]:
"""Nadeau-Bengio paired comparisons, per-fold class counts, hard-label AUROC = balanced accuracy."""
from scipy import stats
ab=pd.read_csv(OUT_DIR/'abl.csv'); ri=pd.read_csv(OUT_DIR/'res_imp.csv')
def nb(a,b,n_tr,n_te):
    d=(a-b).values; n=len(d); v=d.var(ddof=1); se=np.sqrt((1/n+n_te/n_tr)*v)
    t=d.mean()/se if se>0 else 0; q=stats.t.ppf(.975,n-1)
    return d.mean(),2*stats.t.sf(abs(t),n-1),(d.mean()-q*se,d.mean()+q*se)
models=sorted(ab.model.unique()); rows=[]
for m in models:
    key=lambda t: ab[(ab.model==m)&(ab.tag=='abl_'+t)].sort_values(['rep','fold']).reset_index(drop=True)
    for A,B in [('both','values_only'),('both','indicators_only'),('indicators_only','values_only'),('values_only','demographics'),('indicators_only','demographics')]:
        for met in ['auroc','auprc']:
            a,b=key(A),key(B); mu,p,ci=nb(a[met],b[met],a.ntr.mean(),a.nte.mean())
            rows.append(dict(model=m,contrast=f'{A} - {B}',metric=met,delta=mu,lo=ci[0],hi=ci[1],p_nb=p))
pd.DataFrame(rows).to_csv(OUT_DIR/'paired.csv',index=False)
rows=[]
for m in models:
    base=ri[(ri.model==m)&(ri.tag=='res_weights')].reset_index(drop=True)
    for r in ['none','smote','under']:
        o=ri[(ri.model==m)&(ri.tag=='res_'+r)].reset_index(drop=True); mu,p,ci=nb(o.auroc,base.auroc,base.ntr.mean(),base.nte.mean())
        rows.append(dict(model=m,contrast=f'{r} - weights',delta=mu,lo=ci[0],hi=ci[1],p_nb=p))
pd.DataFrame(rows).to_csv(OUT_DIR/'paired_res.csv',index=False)
fc=ab[(ab.tag=='abl_both')&(ab.model==models[0])][['rep','fold','ntr','nte','pos_tr','pos_te','pts_te','overlap']].copy()
fc['neg_te']=fc.nte-fc.pos_te; fc['neg_tr']=fc.ntr-fc.pos_tr; fc.to_csv(OUT_DIR/'fold_counts.csv',index=False); print(fc.agg(['min','max']))
hl=ab[ab.tag=='abl_both'].groupby('model')[['auroc','auroc_hard','bacc','auprc']].mean(); hl.to_csv(OUT_DIR/'hardlabel.csv'); print(hl.round(4))


     rep  fold   ntr  nte  pos_tr  pos_te  pts_te  overlap  neg_te  neg_tr
min    0     0  1701  411     164      34     338        0     362    1515
max    2     4  1744  454     186      56     347        0     420    1573
        auroc  auroc_hard    bacc   auprc
model                                    
LR     0.6080      0.5752  0.5752  0.1807
MLP    0.5999      0.5649  0.5649  0.1873
RF     0.5938      0.5421  0.5421  0.1811
XGB    0.5653      0.5586  0.5586  0.1634


## 8. Step 4 – pooled CIs, calibration, operating points, hospital-grouped validation, landmark, feature table

In [9]:
"""Pooled OOF CIs (patient-cluster bootstrap), calibration, alert-rate operating points,
hospital-grouped validation, landmark analysis, feature summary table."""
import statsmodels.api as sm
from sklearn.model_selection import GroupKFold
p=pd.read_csv(EICU_RAW_DIR/'patient.csv.gz').set_index('patientunitstayid')
fo=pd.read_csv(OUT_DIR/'first_offsets.csv',index_col=0).iloc[:,0]
f['hosp']=f.patientunitstayid.map(p.hospitalid); f['los']=f.patientunitstayid.map(p.unitdischargeoffset); f['first']=f.patientunitstayid.map(fo)
allc=SETS['both']
def oof(idx,yy,gg,cols,name,R):
    P=np.zeros((R,len(idx)))
    for r in range(R):
        for a,b in StratifiedGroupKFold(5,shuffle=True,random_state=42+r).split(idx,yy,gg):
            ff=f.iloc[idx]; Xa=ff.iloc[a][cols].astype(float); Xb=ff.iloc[b][cols].astype(float); m_=Xa.median()
            P[r,b]=fit_pred(name,Xa.fillna(m_).values,Xb.fillna(m_).values,yy[a],'weights',r,ga=gg[a])
    return P.mean(0)
def ci_boot(yy,pp,gg,B=1000,seed=0):
    rng=np.random.default_rng(seed); ug=np.unique(gg); idx={u:np.where(gg==u)[0] for u in ug}; A=[];Q=[]
    for _ in range(B):
        s=np.concatenate([idx[u] for u in rng.choice(ug,len(ug))])
        if yy[s].sum()==0: continue
        A.append(roc_auc_score(yy[s],pp[s])); Q.append(average_precision_score(yy[s],pp[s]))
    return np.percentile(A,[2.5,97.5]),np.percentile(Q,[2.5,97.5])
idx=np.arange(len(f)); rows=[]; allp={}
for m in MODELS:
    pp=oof(idx,y,g,allc,m,N_REPEATS); allp[m]=pp; a,q=ci_boot(y,pp,g)
    rows.append(dict(model=m,auroc=roc_auc_score(y,pp),auroc_lo=a[0],auroc_hi=a[1],auprc=average_precision_score(y,pp),auprc_lo=q[0],auprc_hi=q[1],prev=y.mean())); print(rows[-1],flush=True)
pd.DataFrame(rows).to_csv(OUT_DIR/'pooled.csv',index=False)
cal=[]
for m in MODELS:
    pp=np.clip(allp[m],1e-6,1-1e-6); lg=np.log(pp/(1-pp)); c=sm.Logit(y,sm.add_constant(lg)).fit(disp=0)
    cal.append(dict(model=m,slope=c.params[1],intercept=c.params[0],brier=np.mean((allp[m]-y)**2),brier_null=np.mean((y.mean()-y)**2)))
pd.DataFrame(cal).to_csv(OUT_DIR/'calib.csv',index=False); print(pd.DataFrame(cal).round(3))
op=[]
for frac in [0.05,0.10,0.20,0.30]:
    for m in MODELS:
        pm=allp[m]; al=pm>=np.quantile(pm,1-frac); tp=(al&(y==1)).sum()
        op.append(dict(model=m,alert_rate=frac,sens=tp/y.sum(),ppv=tp/al.sum(),spec=((~al)&(y==0)).sum()/(y==0).sum(),lift=(tp/al.sum())/y.mean()))
pd.DataFrame(op).to_csv(OUT_DIR/'operating.csv',index=False)
res=[]
for m in MODELS:
    P=np.zeros(len(f))
    for a,b in GroupKFold(5).split(f,y,f.hosp):
        Xa=f.iloc[a][allc].astype(float);Xb=f.iloc[b][allc].astype(float);mm=Xa.median()
        P[b]=fit_pred(m,Xa.fillna(mm).values,Xb.fillna(mm).values,y[a],'weights',0,ga=g[a])
    a_,_=ci_boot(y,P,f.hosp.values,B=500); res.append(dict(model=m,auroc=roc_auc_score(y,P),lo=a_[0],hi=a_[1],auprc=average_precision_score(y,P))); print(res[-1],flush=True)
pd.DataFrame(res).to_csv(OUT_DIR/'hospital_grouped.csv',index=False)
keep=(f.los>=1440)&((~f.Target)|(f['first']>1440)); sub=f[keep].reset_index(drop=True); ys=sub.Target.astype(int).values; gs=sub.uniquepid.values
print('landmark n',len(sub),'pos',ys.sum()); lm=[]
for m in MODELS:
    for name,cols in [('both',allc),('indicators',SETS['indicators_only']),('values',SETS['values_only']),('demographics',demo)]:
        A=[]
        for r in range(5):
            for a,b in StratifiedGroupKFold(5,shuffle=True,random_state=42+r).split(sub,ys,gs):
                Xa=sub.iloc[a][cols].astype(float);Xb=sub.iloc[b][cols].astype(float);mm=Xa.median()
                pr=fit_pred(m,Xa.fillna(mm).values,Xb.fillna(mm).values,ys[a],'weights',r,ga=gs[a])
                if ys[b].sum()>0: A.append(roc_auc_score(ys[b],pr))
        lm.append(dict(model=m,set=name,auroc=np.mean(A),sd=np.std(A,ddof=1),n=len(sub),pos=int(ys.sum())))
pd.DataFrame(lm).to_csv(OUT_DIR/'landmark.csv',index=False); print(pd.DataFrame(lm).round(3))
rows=[]
for c in [c for c in f.columns if c.endswith(('_mean','_measured','_given'))]+demo:
    x=pd.to_numeric(f[c],errors='coerce'); rows.append(dict(feature=c,type='continuous' if c.endswith('_mean') or c in('age','admissionweight') else 'binary',missing_pct=100*x.isna().mean(),mean=x.mean(),min=x.min(),max=x.max(),mean_pos=x[y==1].mean(),mean_neg=x[y==0].mean()))
pd.DataFrame(rows).to_csv(OUT_DIR/'feature_summary.csv',index=False)


{'model': 'LR', 'auroc': np.float64(0.6109960065774019), 'auroc_lo': np.float64(0.5692702564346394), 'auroc_hi': np.float64(0.6514112682171671), 'auprc': np.float64(0.16349289192909222), 'auprc_lo': np.float64(0.13260294573606618), 'auprc_hi': np.float64(0.2064397907215557), 'prev': np.float64(0.10208816705336426)}
{'model': 'RF', 'auroc': np.float64(0.5984190744655861), 'auroc_lo': np.float64(0.5573203110826674), 'auroc_hi': np.float64(0.6405669786487102), 'auprc': np.float64(0.16706638990036007), 'auprc_lo': np.float64(0.1319351437918802), 'auprc_hi': np.float64(0.21550457313610438), 'prev': np.float64(0.10208816705336426)}
{'model': 'XGB', 'auroc': np.float64(0.5729668780831572), 'auroc_lo': np.float64(0.5286133909548357), 'auroc_hi': np.float64(0.6162516250744666), 'auprc': np.float64(0.15130488421507926), 'auprc_lo': np.float64(0.12263787590014873), 'auprc_hi': np.float64(0.19674916271306636), 'prev': np.float64(0.10208816705336426)}
{'model': 'MLP', 'auroc': np.float64(0.62025135

## 9. Download all results
Send the zip back.

In [10]:
!cd /content && zip -qr outputs.zip outputs
from google.colab import files; files.download('/content/outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>